<!--
SPDX-FileCopyrightText: 2026 Ernesto Perez
SPDX-License-Identifier: MPL-2.0
-->

# Switch count under system-matrix recomputation

With precomputed system matrices the MNA solver builds one matrix per switch
state combination and selects it through a `std::bitset` of `SWITCH_NUM` bits,
so more than `SWITCH_NUM` (64) switches cannot be represented. With
system-matrix recomputation there is a single matrix and no such bitset, so the
switch count must not be limited there.

This notebook checks four cases on a chain of switches, where every switch
closes onto a node with a resistive load:

1. 70 DP switches with recomputation `Enabled`: the simulation runs, and
   opening the last switch removes the voltage of the last node.
2. 70 DP switches with recomputation `Disabled`: the simulation is rejected,
   because the precomputed path cannot hold 2^70 matrices.
3. 3 DP switches with recomputation `Disabled` and `Enabled`: both run and give
   the same result, so the precomputed path is unchanged.
4. 70 EMT switches with recomputation `Enabled`: the real-valued solver runs
   too, since the switch-count check exists in both solver specializations.

All checks are implemented as assertions.


In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

import dpsimpy


Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
Solver = dpsimpy.Solver
Mode = dpsimpy.SystemMatrixRecomputationMode

dp = dpsimpy.dp
emt = dpsimpy.emt

log_root = Path("logs/SwitchCountRecomputation")

system_frequency = 50.0
time_step = 1e-4
final_time = 0.1
open_time = 0.05


def prepare_run(name):
    run_dir = log_root / name
    shutil.rmtree(run_dir, ignore_errors=True)
    Logger.set_log_dir(str(run_dir))
    return run_dir


def read_result(run_dir, name):
    path = run_dir / f"{name}.csv"
    assert path.exists(), f"Missing simulation result: {path}"
    return pd.read_csv(path, skipinitialspace=True).rename(columns=str.strip)

## Switch chain

A voltage source feeds the chain through a short RL line. Every switch closes
onto a node with a 100 Ω load to ground. The last switch opens at
`open_time`, and the logger records the voltage of the last node.


In [ ]:
def build_switch_chain(name, switch_count, mode, domain=Domain.DP):
    module = dp if domain == Domain.DP else emt
    ph1 = module.ph1
    gnd = module.SimNode.gnd
    n_source = module.SimNode("nSource", PhaseType.Single)
    n_line = module.SimNode("nLine", PhaseType.Single)
    nodes = [gnd, n_source, n_line]

    source = ph1.VoltageSource("Source")
    source.set_parameters(V_ref=complex(1000.0, 0.0), f_src=system_frequency)
    source.connect([gnd, n_source])

    line_resistor = ph1.Resistor("LineR")
    line_resistor.set_parameters(1.0)
    line_resistor.connect([n_source, n_line])

    n_chain = module.SimNode("n0", PhaseType.Single)
    nodes.append(n_chain)
    line_inductor = ph1.Inductor("LineL")
    line_inductor.set_parameters(1e-3)
    line_inductor.connect([n_line, n_chain])

    components = [source, line_resistor, line_inductor]
    switches = []
    previous = n_chain
    for index in range(1, switch_count + 1):
        node = module.SimNode(f"n{index}", PhaseType.Single)
        switch = ph1.Switch(f"Switch{index}")
        switch.set_parameters(1e9, 1e-3, True)
        switch.connect([previous, node])
        load = ph1.Resistor(f"Load{index}")
        load.set_parameters(100.0)
        load.connect([node, gnd])
        nodes.append(node)
        components += [switch, load]
        switches.append(switch)
        previous = node

    system = SystemTopology(system_frequency, nodes, components)

    logger = Logger(name)
    logger.log_attribute("v_last", previous.attr("v"))

    sim = Simulation(name, dpsimpy.LogLevel.info)
    sim.set_system(system)
    sim.set_domain(domain)
    sim.set_solver(Solver.MNA)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.set_system_matrix_recomputation_mode(mode)
    sim.add_logger(logger)
    sim.add_event(dpsimpy.event.SwitchEvent(open_time, switches[-1], False))
    return sim


def last_node_voltage(result):
    if "v_last" in result:  # EMT: the instantaneous value
        magnitude = result["v_last"].abs()
    else:  # DP: the magnitude of the dynamic phasor
        magnitude = np.hypot(result["v_last.re"], result["v_last.im"])
    # From 20 ms: the logger's first sample, at t = 0, precedes the initialization.
    before = magnitude[
        (result["time"] > 0.02) & (result["time"] < open_time - 10 * time_step)
    ]
    after = magnitude[result["time"] > open_time + 10 * time_step]
    return before, after


def run_case(name, switch_count, mode, domain=Domain.DP):
    run_dir = prepare_run(name)
    sim = build_switch_chain(name, switch_count, mode, domain)
    sim.run()
    return read_result(run_dir, name)

## 1. More switches than `SWITCH_NUM` with recomputation enabled


In [ ]:
many = run_case("SwitchCount_70_Enabled", 70, Mode.Enabled)
before, after = last_node_voltage(many)

# The DP magnitude is steady, so the whole window before the opening must hold it.
assert before.min() > 100.0, "The last node must carry a voltage before opening."
assert (
    after.max() < 1e-3 * before.max()
), "Opening the last switch must remove its voltage."

print("70 switches with recomputation: ok")

## 2. More switches than `SWITCH_NUM` with recomputation disabled


In [ ]:
prepare_run("SwitchCount_70_Disabled")
sim = build_switch_chain("SwitchCount_70_Disabled", 70, Mode.Disabled)
rejected = False
try:
    sim.run()
except RuntimeError:
    rejected = True

assert rejected, "Precomputed matrices must reject more than SWITCH_NUM switches."

print("70 switches with precomputed matrices: rejected as expected")

## 3. The precomputed path with few switches


In [ ]:
few_disabled = run_case("SwitchCount_3_Disabled", 3, Mode.Disabled)
few_enabled = run_case("SwitchCount_3_Enabled", 3, Mode.Enabled)

before, after = last_node_voltage(few_disabled)
assert before.max() > 100.0
assert after.max() < 1e-3 * before.max()

np.testing.assert_allclose(
    few_disabled.to_numpy(),
    few_enabled.to_numpy(),
    rtol=1e-8,
    atol=1e-8,
    err_msg="Precomputed and recomputed results differ for 3 switches.",
)

print("3 switches: precomputed and recomputed results agree")

## 4. More switches than `SWITCH_NUM` in the EMT domain


In [ ]:
many_emt = run_case("SwitchCount_70_Enabled_EMT", 70, Mode.Enabled, Domain.EMT)
before, after = last_node_voltage(many_emt)

assert before.max() > 100.0, "The last EMT node must carry a voltage before opening."
assert (
    after.max() < 1e-3 * before.max()
), "Opening the last EMT switch must remove its voltage."

print("70 EMT switches with recomputation: ok")